# Tutorial 3 — Enrichment

This tutorial covers the S-LDSC heritability enrichment stage:

1. Building per-factor peak annotations — thin → full format widening
2. The `is_full_annot()` guard and the `force_decimal()` dtype workaround
3. Resolving the LDSC binary with `ensure_ldsc()`
4. Showing what `build_command()` produces (what `run_ldsc()` would call)
5. Reading `.results` files and computing BH q-values

We use synthetic data throughout so the notebook runs in any environment. The actual
`run_ldsc()` call (Section 4) requires real LD reference panels and GWAS summary
statistics; its invocation is shown in a code comment only.

## 0. Shared setup — synthetic plant dataset

In [1]:
from pathlib import Path
import tempfile
import numpy as np
import pandas as pd
import h5py

from scads_drvi.config import Project

TISSUES  = ("leaf", "root", "stem", "flower")
LINEAGES = ("mesophyll", "epidermis", "vasculature")
FINE_TYPES = {
    "mesophyll":   ("meso_1", "meso_2", "meso_3"),
    "epidermis":   ("epi_1",  "epi_2"),
    "vasculature": ("vasc_1", "vasc_2", "vasc_3"),
}
TRAITS    = ("plant_height", "grain_yield")
N_CELLS, N_FACTORS, N_KEPT = 3_000, 12, 9

root = Path(tempfile.mkdtemp())
rng  = np.random.default_rng(0)

proj = Project(root=root, fit="assembly_v3", traits=TRAITS)
arm  = "assembly_v3_topfrac"
arm_dir = proj.enrich_dir(arm)
arm_dir.mkdir(parents=True, exist_ok=True)

dims = [f"dim_{j}" for j in range(N_FACTORS)]
print("Project root:", proj.root)

Project root: /tmp/tmpvm8pbub2


## 1. Annotation files — thin → full format

S-LDSC's `--overlap-annot` flag (required for coefficient z-scores) needs **full
format** annotation files: four identifier columns (`CHR BP SNP CM`) followed by the
annotation column(s). Pipelines built against the Python LDSC tend to write **thin**
files (annotation columns only) because the Python parser drops identifiers silently.

`to_full_annot()` widens a thin file by joining it positionally with the `.bim`; a
length mismatch raises rather than aligning on keys to keep row order intact.

In [2]:
from scads_drvi.enrich.annotations import (
    BIM_COLUMNS, IDENTIFIER_COLUMNS,
    read_bim, is_full_annot, to_full_annot, write_full_annot, force_decimal,
)

print("BIM column order:        ", BIM_COLUMNS)
print("Full-annot identifiers:  ", IDENTIFIER_COLUMNS)

# Create a synthetic .bim — 500 SNPs on chr1
n_snps = 500
bim_path = root / "chr1.bim"
bim_df = pd.DataFrame({
    "CHR": 1,
    "SNP": [f"rs{i:07d}" for i in range(n_snps)],
    "CM":  np.linspace(0.0, 12.3, n_snps),   # starts at 0, goes decimal later
    "BP":  np.arange(1_000, 1_000 + n_snps * 1_000, 1_000),
    "A1":  rng.choice(list("ACGT"), size=n_snps),
    "A2":  rng.choice(list("ACGT"), size=n_snps),
})
bim_df.to_csv(bim_path, sep="\t", header=False, index=False)
bim = read_bim(bim_path)
print(f"\nBIM: {len(bim)} variants")
bim.head()

BIM column order:         ('CHR', 'SNP', 'CM', 'BP', 'A1', 'A2')
Full-annot identifiers:   ('CHR', 'BP', 'SNP', 'CM')

BIM: 500 variants


,CHR,SNP,CM,BP,A1,A2
0,1,rs0000000,0.000000,1000,T,A
1,1,rs0000001,0.024649,2000,G,C
2,1,rs0000002,0.049299,3000,G,G
3,1,rs0000003,0.073948,4000,C,C
4,1,rs0000004,0.098597,5000,C,G


In [3]:
# Thin annotation — just the annotation column
thin_annot = pd.DataFrame({"k1L2_0": rng.integers(0, 2, size=n_snps).astype(float)})
print("is_full_annot(thin): ", is_full_annot(thin_annot.columns))

# Widen to full format
full_annot = to_full_annot(thin_annot, bim)
print("is_full_annot(full): ", is_full_annot(full_annot.columns))
print("Full annot shape:    ", full_annot.shape)
full_annot.head()

is_full_annot(thin):  False
is_full_annot(full):  True
Full annot shape:     (500, 5)


,CHR,BP,SNP,CM,k1L2_0
0,1,1000,rs0000000,0.000000,1.0
1,1,2000,rs0000001,0.024649,0.0
2,1,3000,rs0000002,0.049299,0.0
3,1,4000,rs0000003,0.073948,1.0
4,1,5000,rs0000004,0.098597,0.0


In [4]:
# write_full_annot: widen and gzip-compress to disk in one call
annot_dir = proj.annot_dir(arm)
annot_dir.mkdir(parents=True, exist_ok=True)

out_path = annot_dir / "k1.1.annot.gz"
write_full_annot(out_path, thin_annot, bim, decimal=True)
print("Written:", out_path)
print("Size:   ", out_path.stat().st_size, "bytes")

# Verify it reads back cleanly
check = pd.read_csv(out_path, sep="\t")
print("Columns:", list(check.columns))
print("Rows:   ", len(check))

Written: /tmp/tmpvm8pbub2/data/annotations/assembly_v3_topfrac/k1.1.annot.gz
Size:    5443 bytes
Columns: ['CHR', 'BP', 'SNP', 'CM', 'k1L2_0']
Rows:    500


### `force_decimal()` — fixing the Rust reader's type inference

The Rust LDSC reader infers column types from a leading sample. When the genetic-map
distance column (`CM`) is 0.0 for the first few hundred rows it infers `Int64`, then
fails on the first non-integer value with a CSV parse error. `force_decimal()` rewrites
every numeric column that contains integers using explicit decimal notation (`0` → `0.0`)
so the reader always infers float.

In [5]:
# Demonstrate force_decimal on the first few rows (CM starts at 0)
sample = full_annot.head(10).copy()
print("CM before force_decimal:", sample["CM"].tolist())

fixed = force_decimal(sample)
print("CM after  force_decimal:", fixed["CM"].tolist())

# In the pipeline this is applied before writing baseline LD annotations:
# baseline = full_annot_dir / f"baselineLD.{chrom}.annot.gz"
# if baseline.exists():
#     df = pd.read_csv(baseline, sep="\t")
#     force_decimal(df).to_csv(baseline, sep="\t", index=False)

CM before force_decimal: [0.0, 0.0246492985971943, 0.0492985971943887, 0.0739478957915831, 0.0985971943887775, 0.1232464929859719, 0.1478957915831663, 0.1725450901803607, 0.1971943887775551, 0.2218436873747495]
CM after  force_decimal: [0.0, 0.0246492985971943, 0.0492985971943887, 0.0739478957915831, 0.0985971943887775, 0.1232464929859719, 0.1478957915831663, 0.1725450901803607, 0.1971943887775551, 0.2218436873747495]


## 2. Write the `factor_map.tsv`

`factor_map.tsv` maps each dimension to its annotation index. `load_labels()` reads it
and builds the `FactorLabels` object used downstream by `read_results()` and all
visualisation functions.

In [6]:
from scads_drvi.labels import load_labels

kept_flags = [True] * N_KEPT + [False] * (N_FACTORS - N_KEPT)
fmap = pd.DataFrame({
    "dim":         dims,
    "vanished":    [False] * N_FACTORS,
    "kept":        kept_flags,
    "drop_reason": ["" if k else "annot_too_small" for k in kept_flags],
    "annot_index": [i + 1 if k else None for i, k in enumerate(kept_flags)],
})
fmap_path = arm_dir / "factor_map.tsv"
fmap.to_csv(fmap_path, sep="\t", index=False)

labels = load_labels(fmap_path, model=arm)
print("Kept dims:  ", labels.kept_dims[:4], "...")
print("annot2dim:  ", dict(list(labels.annot2dim.items())[:3]), "...")
print("is_split:   ", labels.is_split)

Kept dims:   ('dim_0', 'dim_1', 'dim_2', 'dim_3') ...
annot2dim:   {'k1': 'dim_0', 'k2': 'dim_1', 'k3': 'dim_2'} ...
is_split:    False


## 3. Resolve the LDSC binary with `ensure_ldsc()`

`ensure_ldsc` searches in order:
1. `$SCADS_DRVI_LDSC` environment variable
2. XDG cache (`~/.cache/scads_drvi/ldsc/v0.5.0/`)
3. Downloads from GitHub Releases and verifies SHA-256 (~6 MB, once per machine)

`allow_download=False` returns whatever is already cached, or raises.

In [7]:
from scads_drvi.enrich.binary import ensure_ldsc, build_command, LDSC_VERSION

try:
    binary = ensure_ldsc(version=LDSC_VERSION, allow_download=True)
    print("LDSC binary:", binary.path)
    print("Version:    ", binary.version)
    print("Source:     ", binary.source)
    have_binary = True
except Exception as exc:
    print(f"Binary not available ({exc}); skipping build_command demo.")
    have_binary = False

LDSC binary: /home/mike.cuoco/.cache/scads_drvi/ldsc/v0.5.0/ldsc
Version:     0.5.0
Source:      cache


## 4. Preview the LDSC command with `build_command()`

`build_command()` returns the list of strings that `run_ldsc()` would pass to
`subprocess.run`. Inspecting it lets you verify the flags before committing a long
queue of jobs.

In [8]:
if have_binary:
    sumstats_path = proj.data / "sumstats" / "plant_height.sumstats.gz"
    ref_ld        = proj.ldsc_ref or Path("/data/ldsc/ref/chr")   # placeholder

    # build_command(subcommand, options, *, binary=...)
    # Option keys use underscores; the function adds "--" and converts _ to -
    cmd = build_command(
        "h2",
        {
            "h2":                str(sumstats_path),
            "ref_ld_chr":        str(annot_dir / "k1/"),
            "w_ld_chr":          str(ref_ld),
            "out":               str(arm_dir / "results" / "plant_height" / "k1"),
            "overlap_annot":     True,
            "print_coefficients": True,
        },
        binary=binary,
    )
    print("Command preview:")
    for part in cmd:
        print(" ", part)
else:
    print("(skipped — binary unavailable)")

Command preview:
  LdscBinary(path=PosixPath('/home/mike.cuoco/.cache/scads_drvi/ldsc/v0.5.0/ldsc'), version='0.5.0', source='cache')
  h2
  --h2
  /tmp/tmpvm8pbub2/data/sumstats/plant_height.sumstats.gz
  --ref-ld-chr
  /tmp/tmpvm8pbub2/data/annotations/assembly_v3_topfrac/k1
  --w-ld-chr
  /data/ldsc/ref/chr
  --out
  /tmp/tmpvm8pbub2/data/enrich/assembly_v3_topfrac/results/plant_height/k1
  --overlap-annot
  --print-coefficients


## 5. Create synthetic `.results` files and read them back

In production, `run_ldsc()` writes one `.results` file per (factor, trait). Here we
write them by hand using the same column layout so we can demonstrate `read_results()`,
`add_fdr()`, and `significant()` without needing actual LD reference panels.

In [9]:
# Write synthetic .results files
results_root = arm_dir / "results"
annot2dim = labels.annot2dim   # {"k1": "dim_0", "k2": "dim_1", ...}

for ti, trait in enumerate(TRAITS):
    td = results_root / trait
    td.mkdir(parents=True, exist_ok=True)
    for i, (annot, dim) in enumerate(annot2dim.items()):
        z = 4.5 - 0.5 * i + 0.3 * ti   # decreasing z-scores
        pd.DataFrame({
            "Category":              [f"{annot}L2_0"],
            "Coefficient":           [1e-8 * z],
            "Coefficient_std_error": [1e-8],
            "Coefficient_z-score":   [z],
            "n_categories":          [40],
            "total_h2":              [0.11],
        }).to_csv(td / f"{annot}.results", sep="\t", index=False)

print(f"Wrote {N_KEPT} .results files for each of {len(TRAITS)} traits.")

Wrote 9 .results files for each of 2 traits.


In [10]:
from scads_drvi.enrich.ldsc import read_results, results_files

# results_files: quick listing for one trait
print("Files for plant_height:")
for name, path in list(results_files(results_root / "plant_height").items())[:4]:
    print(f"  {name} -> {path.name}")

# read_results: aggregate across all traits, compute BH q-values within each
results = read_results(
    results_root,
    traits=list(TRAITS),
    labels=labels,
    strict=True,  # raises if any expected .results file is missing
    fdr=True,     # adds p_1tailed and fdr_q columns
    by="trait",   # BH correction applied within each trait independently
)
print("\nShape:", results.shape)
print("Columns:", results.columns.tolist())
results.head()

Files for plant_height:
  k1 -> k1.results
  k2 -> k2.results
  k3 -> k3.results
  k4 -> k4.results



Shape: (18, 12)
Columns: ['Category', 'Coefficient', 'Coefficient_std_error', 'Coefficient_z-score', 'n_categories', 'total_h2', 'trait', 'annot', 'dim', 'display', 'p_1tailed', 'fdr_q']


,Category,Coefficient,Coefficient_std_error,Coefficient_z-score,n_categories,total_h2,trait,annot,dim,display,p_1tailed,fdr_q
0,k1L2_0,4.500000e-08,1.000000e-08,4.5,40,0.11,plant_height,k1,dim_0,dim_0,0.000003,0.000031
1,k2L2_0,4.000000e-08,1.000000e-08,4.0,40,0.11,plant_height,k2,dim_1,dim_1,0.000032,0.000143
2,k3L2_0,3.500000e-08,1.000000e-08,3.5,40,0.11,plant_height,k3,dim_2,dim_2,0.000233,0.000698
3,k4L2_0,3.000000e-08,1.000000e-08,3.0,40,0.11,plant_height,k4,dim_3,dim_3,0.001350,0.003037
4,k5L2_0,2.500000e-08,1.000000e-08,2.5,40,0.11,plant_height,k5,dim_4,dim_4,0.006210,0.011177


## 6. Compute BH q-values and test significance

`add_fdr()` converts a two-sided z-score to a one-tailed p-value and then applies
Benjamini-Hochberg within each group defined by `by`. `significant()` applies a
threshold to the resulting q-values.

In [11]:
from scads_drvi.stats import add_fdr, significant

# add_fdr was already applied inside read_results; shown here standalone
results_no_fdr = results.drop(columns=["p_1tailed", "fdr_q"])
results_with_q = add_fdr(results_no_fdr, by="trait")
print("Columns after add_fdr:", [c for c in results_with_q.columns if "fdr" in c or "p_1" in c])

# significant() returns a boolean Series
print("\nSignificant factors per trait (q < 0.05):")
for trait in TRAITS:
    subset = results[results["trait"] == trait]
    n_sig  = significant(subset, alpha=0.05).sum()
    print(f"  {trait}: {n_sig} / {len(subset)}")

Columns after add_fdr: ['p_1tailed', 'fdr_q']

Significant factors per trait (q < 0.05):
  plant_height: 6 / 9
  grain_yield: 7 / 9


## Calling `run_ldsc()` with a real dataset

With real LD reference panels and GWAS summary statistics, the enrichment loop is:

```python
from scads_drvi.enrich.binary import ensure_ldsc, run_ldsc, LDSC_VERSION

binary = ensure_ldsc(version=LDSC_VERSION, allow_download=True)

for trait in proj.traits:
    sumstats = proj.data / "sumstats" / f"{trait}.sumstats.gz"
    for annot, dim in labels.annot2dim.items():
        result_dir = proj.results_dir(arm, trait)
        result_dir.mkdir(parents=True, exist_ok=True)
        run_ldsc(
            "h2",
            {
                "h2":                 str(sumstats),
                "ref_ld_chr":         str(full_annot_dir / f"{annot}/"),
                "w_ld_chr":           str(proj.ldsc_ref / "weights_hm3_no_hla/"),
                "out":                str(result_dir / annot),
                "overlap_annot":      True,
                "print_coefficients": True,
            },
            binary=binary,
            threads=1,
            log_fn=print,
        )
```

See the [S-LDSC documentation](https://github.com/bulik/ldsc) and
[sharifhsn/ldsc](https://github.com/sharifhsn/ldsc) for how to obtain the LD reference
panels and munge GWAS summary statistics into `.sumstats.gz` format.